# Microsoft Foundry Local 
## Running Large Language Models Locally with Foundry Local

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook is a **hands-on demonstration of running and interacting with Large Language Models (LLMs) locally** using **Foundry Local** and an **OpenAI-compatible API**.  
It showcases how to discover, load, and interact with models entirely on-device cloud services.

The notebook is intended as a **practical reference and demo** for:
- Offline / disconnected environments
- Edge or on-device AI scenarios
- Enterprise and customer demonstrations
- Prompt engineering and API pattern exploration

---

## 📦 Environment & Setup

- Python **3.12**
- Common data and utility libraries: `pandas`, `matplotlib`, `requests`, `statistics`
- Foundry Local runtime managed via `FoundryLocalManager`
- Local OpenAI-compatible endpoint (no API key required for local usage)

The notebook starts the Foundry Local service, validates its status, and exposes:
- Service URI and API endpoint
- Local model cache location

---

## 🧩 Model Catalog & Management

- Lists all available local model variants from the Foundry catalog
- Converts model metadata into a clean `pandas.DataFrame` including:
  - Alias and model ID
  - Device type (CPU / GPU)
  - Execution provider
  - Model size
  - License and supported tasks
  - Tool-calling support
- Supports filtering models by alias (e.g. `phi-4`)
- Demonstrates loading a specific model into memory

---

## 💬 Chat Completions (OpenAI-Compatible)

### Synchronous Chat Completion
- Sends a prompt and receives the full response in one call
- Illustrates basic question–answer usage
- Inspects response and token usage metadata when available

### Streaming Chat Completion
- Streams tokens incrementally as they are generated
- Measures throughput (tokens per second)
- Ideal for real-time UIs and low-latency user experiences

---

## 🎛️ Prompt Engineering & Output Control

### System Prompts
- Control assistant behavior, tone, and format
- Enforce structured outputs (e.g. JSON-only responses)

### Examples Included
- **Structured JSON extraction** (entity extraction from text)
- **Persona-based responses** (e.g. friendly data scientist, playful explanations)
- **Temperature tuning** for deterministic vs. creative outputs

---

## 🔁 Multi-Turn Conversations

- Demonstrates how to maintain conversation context
- Uses a shared `messages` array across multiple turns
- Example: concise math tutor answering follow-up questions
- Highlights that context is managed client-side

---

## 🌐 Direct REST API Usage

- Calls the Foundry Local REST API directly using `requests`
- Bypasses the OpenAI SDK entirely
- Shows raw request / response structure
- Confirms OpenAI API compatibility at the HTTP level

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/foundry-local/reference/reference-sdk-current?pivots=programming-language-python)

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

## Setup

## Current Foundry Local SDK setup

These examples use the **1.2.4 Python SDK** (`foundry_local_sdk`), not the legacy
`foundry_local` package. Use Python 3.11+ (validated with Python 3.12).
Uncomment the installation command for your OS, run it once, then restart the kernel.
Do not install both SDK distributions in the same environment.

The SDK hosts its own runtime; installing or upgrading the CLI does not upgrade the
Python SDK. Catalog entries are models with hardware-specific `variants`; metadata
is available through `model.info`, with device/provider under `info.runtime`.
Aliases are case-sensitive. `model.download()` downloads the selected variant;
`model.load()` loads it into this SDK runtime.

By default, the SDK uses an application-specific cache. To reuse an existing cache,
set `FOUNDRY_MODEL_CACHE_DIR` before starting the kernel (the CLI shows its cache with
`foundry cache location`). An optional `FOUNDRY_MODEL_ALIAS` selects a different
model for notebooks 01-05; notebook 04 requires Mistral's user/assistant prompt format,
and notebook 05 requires a model that supports tools.

CPU inference works without extra provider downloads. For supported GPU/NPU
acceleration, run `manager.download_and_register_eps()` before querying the catalog.
Provider support differs between SDK and CLI builds: a cached WebGPU model is not
necessarily loadable by the Windows Python SDK. Select/download a supported CPU
variant if necessary. Model and provider downloads require internet access; inference
is local after setup.

References: [SDK](https://learn.microsoft.com/azure/foundry-local/reference/reference-sdk-current?pivots=programming-language-python),
[OpenAI integration](https://learn.microsoft.com/azure/foundry-local/how-to/how-to-integrate-with-inference-sdks).


The chat requests pass `chat_template_kwargs.enable_thinking=False`, verified with
CLI 0.12.1. This keeps cached Qwen3 alternatives in non-reasoning mode so the
demo's output budget is used for visible answers rather than hidden reasoning.
Templates for non-reasoning models ignore this option.


In [1]:
import datetime
import GPUtil
import json
import matplotlib.pyplot as plt
import os
import pandas as pd
import platform
import psutil
import requests
import sys
import time
import statistics

from foundry_local_sdk import Configuration, FoundryLocalManager
from openai import OpenAI

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 02-Nov-2026 10:06:19


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(
    f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical"
)

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(
            f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)"
        )
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 27.3 GB
- RAM used:      57.2%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  918.3 GB (96.5%)
- Free:  33.3 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 3,817 MB (93%)
- VRAM Free  : 119 MB
- GPU Load   : 59%
- Temperature: 63.0 °C


In [5]:
if FoundryLocalManager.instance is None:
    FoundryLocalManager.initialize(Configuration(
        app_name="foundry-local-notebooks",
        model_cache_dir=os.getenv("FOUNDRY_MODEL_CACHE_DIR"),
    ))
manager = FoundryLocalManager.instance
if not manager.urls:
    manager.start_web_service()
endpoint = manager.urls[0].rstrip("/") + "/v1"
print(f"Endpoint (v1): {endpoint}")
print("Cached model paths:", [m.get_path() for m in manager.catalog.get_cached_models()])

Endpoint (v1): http://127.0.0.1:51674/v1
Cached model paths: ['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']


In [6]:
[m.get_path() for m in manager.catalog.get_cached_models()]

['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']

In [7]:
[m.get_path() for m in manager.catalog.get_cached_models()]

['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']

## Helper

In [8]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": (m.runtime.device_type.value if m.runtime else "unknown"),
        "provider": (m.runtime.execution_provider if m.runtime else "unknown"),
        "size_mb": m.file_size_mb,
        "tools": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

## Models

In [9]:
catalog = [v.info for m in manager.catalog.list_models() for v in m.variants]
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 56


,alias,id,device,provider,size_mb,tools,license,task
0,deepseek-r1-1.5b,deepseek-r1-distill-qwen-1.5b-generic-cpu:4,CPU,CPUExecutionProvider,1966,None,MIT,chat-completion
1,deepseek-r1-14b,deepseek-r1-distill-qwen-14b-generic-cpu:4,CPU,CPUExecutionProvider,11786,None,MIT,chat-completion
2,deepseek-r1-7b,deepseek-r1-distill-qwen-7b-generic-cpu:4,CPU,CPUExecutionProvider,6584,None,MIT,chat-completion
3,deepseek-r1-8b,deepseek-r1-distill-llama-8b-generic-cpu:4,CPU,CPUExecutionProvider,6796,None,MIT,chat-completion
4,gemma-4-e2b-it,gemma-4-e2b-it-generic-cpu:3,CPU,CPUExecutionProvider,6031,True,apache-2.0,vision-language-chat
5,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,None,MIT,chat-completion
6,ministral-3-3b-instruct-2512,ministral-3-3b-instruct-2512-generic-cpu:2,CPU,CPUExecutionProvider,4691,True,apache-2.0,vision-language-chat
7,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:3,CPU,CPUExecutionProvider,4167,None,apache-2.0,chat-completion
8,mistral-nemo-12b-instruct,mistral-nemo-12b-instruct-generic-cpu:1,CPU,CPUExecutionProvider,6978,True,apache-2.0,chat-completion
9,nemotron-3.5-asr-streaming-0.6b,nemotron-3.5-asr-streaming-0.6b-generic-cpu:3,CPU,CPUExecutionProvider,756,None,MIT,automatic-speech-recognition


In [10]:
# Filter by alias keyword
df_catalog[df_catalog["alias"].str.contains("phi-4", case=False, na=False)]

,alias,id,device,provider,size_mb,tools,license,task
17,phi-4,Phi-4-generic-cpu:2,CPU,CPUExecutionProvider,10403,None,MIT,chat-completion
18,phi-4-mini,Phi-4-mini-instruct-generic-cpu:5,CPU,CPUExecutionProvider,4915,True,MIT,chat-completion
19,phi-4-mini-instruct-generic-cpu-dq44,Phi-4-mini-instruct-generic-cpu-dq44:1,CPU,CPUExecutionProvider,4317,True,MIT,chat-completion
20,phi-4-mini-reasoning,Phi-4-mini-reasoning-generic-cpu:3,CPU,CPUExecutionProvider,4628,True,MIT,chat-completion
21,phi-4-reasoning,Phi-4-reasoning-generic-cpu:1,CPU,CPUExecutionProvider,10403,None,MIT,chat-completion


### Let's use the phi-4-mini model
https://techcommunity.microsoft.com/blog/educatordeveloperblog/welcome-to-the-new-phi-4-models---microsoft-phi-4-mini--phi-4-multimodal/4386037

In [11]:
MODEL_ALIAS = os.getenv("FOUNDRY_MODEL_ALIAS", "phi-4-mini")
model = manager.catalog.get_model(MODEL_ALIAS)
if model is None:
    raise ValueError(f"Model alias {MODEL_ALIAS!r} was not found in the catalog.")
print(f"Selected: {model.id} ({model.info.runtime.execution_provider})")
if not model.is_cached:
    model.download()

Selected: Phi-4-mini-instruct-generic-cpu:5 (CPUExecutionProvider)


In [12]:
[m.get_path() for m in manager.catalog.get_cached_models()]

['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']

In [13]:
model.load()
model_info = model.info
print(f"Loaded: {model_info.alias} => {model_info.id}")

Loaded: phi-4-mini => Phi-4-mini-instruct-generic-cpu:5


In [14]:
info = model.info
print(f"Running on: {info.runtime.execution_provider}")
print(f"Device:     {info.runtime.device_type.value}")

Running on: CPUExecutionProvider
Device:     CPU


In [15]:
# Initialize the OpenAI client
client = OpenAI(
    base_url=endpoint,
    api_key="not-needed"  # Not required for local usage
)

## Synchronous Chat Completion

The simplest way to get a response — send a message, get the full response back at once.

In [16]:
response = client.chat.completions.create(
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    model=model_info.id,
    messages=[{
        "role": "user",
        "content": "What is Pi? Explain in 2 sentences."
    }])

print("🤖 Response:")
print(response.choices[0].message.content)

🤖 Response:
Pi (π) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning its decimal representation never ends and never repeats, and it appears in various formulas across mathematics and physics.


In [17]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-34a26a0de3786e60",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning its decimal representation never ends and never repeats, and it appears in various formulas across mathematics and physics.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1793610398,
  "model": "Phi-4-mini-instruct-generic-cpu:5",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 56,
    "prompt_tokens": 13,
    "total_tokens": 69,
    "completion_tokens_details": {
      "accepted_pr

## Streaming Chat Completion

Streaming returns tokens incrementally as they're generated — great for real-time UIs and lower perceived latency.

In [18]:
print("Streaming response:\n")
start = time.perf_counter()
stream = client.chat.completions.create(
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    model=model_info.id,
    messages=[{"role": "user", "content": "Explain quantum computing in 5 lines."}],
    stream=True,
    stream_options={"include_usage": True},
    max_tokens=512,
)
chunk_count = 0
usage = None
for chunk in stream:
    if chunk.usage is not None:
        usage = chunk.usage
    if not chunk.choices:
        continue
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
        chunk_count += 1
elapsed = time.perf_counter() - start
if usage is not None:
    print(f"\n\n{usage.completion_tokens} output tokens in {elapsed:.2f}s "
          f"({usage.completion_tokens / elapsed:.1f} tokens/sec)")
else:
    print(f"\n\n{chunk_count} text chunks in {elapsed:.2f}s; token usage was not returned.")

Streaming response:

Quantum computing leverages the principles of quantum mechanics to process information. Unlike classical bits, quantum bits or qubits can exist in multiple states simultaneously, thanks to superposition. Quantum entanglement allows qubits that are entangled to be connected in such a way that the state of one (whether spin, position, etc.) instantaneously influences the state of the other, even across large distances. This enables quantum computers to solve certain problems much faster than classical computers by performing many calculations at once. Quantum computing has the potential to revolutionize fields like cryptography, optimization, and simulation of complex systems. However, practical and scalable quantum computing is still a significant challenge.

133 output tokens in 18.06s (7.4 tokens/sec)


In [19]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-34a26a0de3786e60",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Pi (\u03c0) is a mathematical constant representing the ratio of a circle's circumference to its diameter, approximately equal to 3.14159. It is an irrational number, meaning its decimal representation never ends and never repeats, and it appears in various formulas across mathematics and physics.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1793610398,
  "model": "Phi-4-mini-instruct-generic-cpu:5",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 56,
    "prompt_tokens": 13,
    "total_tokens": 69,
    "completion_tokens_details": {
      "accepted_pr

## System Prompts

Use system messages to control the model's behavior, persona, and output format.

In [20]:
# Example 1: JSON output mode
response = client.chat.completions.create(
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    model=model_info.id,
    messages=[{
        "role":
        "system",
        "content":
        "You are a helpful data extraction assistant. "
        "Always respond with valid JSON only, no markdown."
    }, {
        "role":
        "user",
        "content":
        "Extract the key entities from this sentence: "
        "'Marie Curie won the Nobel Prize in Physics in 1903 in Stockholm.'"
    }],
    temperature=0.1  # Low temperature for deterministic output
)

print(response.choices[0].message.content)

{"entities": [{"entity": "Marie Curie", "type": "PERSON"}, {"entity": "Nobel Prize in Physics", "type": "RECOGNITION"}, {"entity": "1903", "type": "DATE"}, {"entity": "Stockholm", "type": "LOCATION"}]}


In [21]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-cd05005d6a6f3348",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"entities\": [{\"entity\": \"Marie Curie\", \"type\": \"PERSON\"}, {\"entity\": \"Nobel Prize in Physics\", \"type\": \"RECOGNITION\"}, {\"entity\": \"1903\", \"type\": \"DATE\"}, {\"entity\": \"Stockholm\", \"type\": \"LOCATION\"}]}",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1793610425,
  "model": "Phi-4-mini-instruct-generic-cpu:5",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 63,
    "prompt_tokens": 48,
    "total_tokens": 111,
    "completion_tokens_details": {
      "accepted_prediction_tokens": null,
      "audio_tokens": null,
      "rea

In [22]:
# Example 2: Persona / role-play
response = client.chat.completions.create(
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    model=model_info.id,
    messages=[{
        "role":
        "system",
        "content":
        "You are a friendly datascientist that explains things "
        "using simple text. Keep answers short and fun."
    }, {
        "role":
        "user",
        "content":
        "What is an object detection model in computer vision?"
    }],
    temperature=0.7  # Higher temperature for creativity
)

print(response.choices[0].message.content)

An object detection model in computer vision is a type of artificial intelligence system that can identify and locate objects within an image or video. Think of it like a super-smart camera that not only sees what you're looking at but also tells you what it is and where it is. This helps computers to understand and interact with the world around them.


In [23]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-7de8ac6c28030d68",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "An object detection model in computer vision is a type of artificial intelligence system that can identify and locate objects within an image or video. Think of it like a super-smart camera that not only sees what you're looking at but also tells you what it is and where it is. This helps computers to understand and interact with the world around them.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1793610437,
  "model": "Phi-4-mini-instruct-generic-cpu:5",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 67,
    "prompt_tokens": 36,
    "total_tokens": 1

## Multi-Turn Conversations

Maintain conversation context by passing the full message history. The model has no memory between calls — context lives in the messages array.

In [24]:
# Simulate a multi-turn conversation
conversation = [
    {
        "role": "system",
        "content": "You are a helpful math tutor. Be concise."
    },
]

questions = [
    "What is a derivative in calculus?",
    "Can you give me a simple example?",
    "What about the chain rule?",
]

for q in questions:
    # Add user message
    conversation.append({"role": "user", "content": q})

    # Get response
    response = client.chat.completions.create(
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        model=model_info.id,
                                              messages=conversation,
                                              max_tokens=2000)

    assistant_msg = response.choices[0].message.content

    # Add assistant response to history
    conversation.append({"role": "assistant", "content": assistant_msg})

    print(f"👤 User: {q}")
    print(f"🤖 Assistant: {assistant_msg}")
    print("-" * 60)

print(f"\n📝 Conversation history: {len(conversation)} messages")

👤 User: What is a derivative in calculus?
🤖 Assistant: A derivative in calculus represents the rate at which a function is changing at any given point. It measures how a function's output value changes with respect to changes in its input value. Mathematically, it is the limit of the average rate of change of the function over an interval as the interval approaches zero. It is denoted by f'(x) or dy/dx, where 'f' is the function and 'x' is the input variable. The derivative provides valuable information about the behavior of the function, such as its slope, rate of change, and whether it is increasing or decreasing at a particular point.
------------------------------------------------------------
👤 User: Can you give me a simple example?
🤖 Assistant: Sure! Let's consider the function f(x) = x^2. We want to find the derivative of this function with respect to x.

Using the power rule for differentiation, which states that if f(x) = x^n, then f'(x) = n * x^(n-1), we can find the derivat

In [25]:
print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "id": "chatcmpl-6c3bc0ca315b75ac",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The chain rule is a formula used to find the derivative of a composite function. It states that if you have two functions, say u(x) and f(u), and you want to find the derivative of the composite function f(u(x)), then the derivative can be found by multiplying the derivative of the outer function with respect to the inner function by the derivative of the inner function with respect to x.\n\nMathematically, if y = f(u) and u = u(x), then the derivative of y with respect to x, denoted as dy/dx, can be found using the chain rule as follows:\n\ndy/dx = (dy/du) * (du/dx)\n\nLet's consider an example to illustrate the chain rule:\n\nSuppose we have a function f(u) = u^2 and an inner function u(x) = 3x + 1. We want to find the derivative of the composite function f(u(x)) with respect to x.\n\nFirst, we find the derivatives

## Direct REST API with `requests`

You can also call the Foundry Local REST API directly without the OpenAI SDK.

In [26]:
# Build the request manually
url = endpoint + "/chat/completions"

payload = {
    "model": model_info.id,
    "messages": [{
        "role": "user",
        "content": "What is the speed of light?"
    }],
    "temperature": 0.7,
    "max_tokens": 2000,
    "chat_template_kwargs": {"enable_thinking": False}
}

headers = {"Content-Type": "application/json"}

resp = requests.post(url, headers=headers, json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()

print(f"Status: {resp.status_code}")
print(f"Model: {data['model']}")
print(f"{data['choices'][0]['message']['content']}")

Status: 200
Model: Phi-4-mini-instruct-generic-cpu:5
The speed of light in a vacuum is approximately 299,792,458 meters per second (or about 186,282 miles per second). This is a fundamental constant of nature and is denoted by the symbol "c". The speed of light is the maximum speed at which all energy, matter, and information in the universe can travel. It is a crucial factor in the theory of relativity developed by Albert Einstein.


In [27]:
print(resp)

<Response [200]>


In [28]:
print(data)

{'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': 'The speed of light in a vacuum is approximately 299,792,458 meters per second (or about 186,282 miles per second). This is a fundamental constant of nature and is denoted by the symbol "c". The speed of light is the maximum speed at which all energy, matter, and information in the universe can travel. It is a crucial factor in the theory of relativity developed by Albert Einstein.', 'refusal': None, 'role': 'assistant'}}], 'created': 1793610584, 'id': 'chatcmpl-8f5c6d46632b1241', 'model': 'Phi-4-mini-instruct-generic-cpu:5', 'object': 'chat.completion', 'system_fingerprint': None, 'usage': {'completion_tokens': 85, 'completion_tokens_details': {'reasoning_tokens': 0}, 'prompt_tokens': 10, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 95}}


# What We Learned?

## Overview

This notebook walks through how to use **Microsoft Foundry Local** to run AI models entirely on a Windows machine, powered by ONNX Runtime, while maintaining an OpenAI-compatible API surface.

The end-to-end workflow covers service setup, model discovery and filtering, local caching, model loading and inference, and cleanup with memory management.

---

## 🚀 Why Foundry Local Matters

Hands-on execution revealed several compelling advantages:

- **Local-only inference** — All prompts and data remain on the machine with no network calls required.
- **Zero runtime cost** — Once models are downloaded, inference runs without any API or token charges.
- **Low latency and offline usage** — No cloud round-trips; performance depends solely on local hardware.
- **Cloud-compatible developer experience** — The API follows the OpenAI specification, so existing SDKs and code work by simply switching the endpoint.
- **Optimized for consumer hardware** — Models are pre-optimized via Microsoft Olive for CPU, CUDA GPU, OpenVINO, and WebGPU backends.

---

## 🖥️ Local Environment Insights

The notebook confirmed that Foundry Local:

- Runs smoothly on **Windows 11**.
- Works with **CPU-only setups** as well as mid-range GPUs.
- Automatically selects the **best model variant** based on available hardware.
- Exposes a local REST endpoint on `127.0.0.1` with a dynamically assigned port.
- Stores models in a **local cache** (e.g., `C:\models`).

---

## 🧠 Model Catalog Exploration

Key takeaways from inspecting the catalog:

The catalog contains **multiple variants per model**, each optimized for a different hardware target. Supported model families include Phi-4, Phi-4-mini, Phi-4-mini-reasoning, Phi-3 and Phi-3.5, Qwen 2.5, Mistral, DeepSeek, Whisper (speech-to-text), and gpt-oss-20b.

Models differ along several dimensions: device type (CPU or GPU), execution provider (CPU, CUDA, OpenVINO, WebGPU), size (from a few hundred megabytes to over 12 GB), and task type (chat completion, automatic speech recognition).

Through filtering and visualization, it was straightforward to identify CPU-only models, spot tool-calling–enabled variants, and compare sizes and providers at a glance.

---

## 📦 Cache and Model Lifecycle

Several observations on model management:

- Downloaded models are **cached locally** and reused across sessions.
- Loading a model is **fast once cached**, with no re-download needed.
- **Multiple models** can be loaded simultaneously, memory permitting.

---

## 🤖 Running Inference with the OpenAI SDK

One of the most striking takeaways is how seamless local inference feels:

- Foundry Local exposes an **OpenAI-compatible endpoint**, meaning the standard `openai` Python SDK works unchanged.
- The model ID must be passed as a **string**, not as a model object.
- Switching between models is as simple as changing the `model` parameter.

Example results confirmed that **Phi-4** ran on GPU via `CUDAExecutionProvider`, while **gpt-oss-20b** ran on CPU — and both produced responses identical in structure to cloud-based chat completions.

---

## 🧹 Cleanup and Resource Management

Finally, Foundry Local provides clean resource controls:

- Models can be **unloaded from memory** without deleting them from cache.
- Unloading **frees RAM and VRAM immediately**.
- Cached models remain available for **instant reload** whenever needed.

> Go to the next notebook